# Week 4 · Day 11 — From Notebook to Application

**Course:** IPAM USL 5-Week Short Course: Introduction to Artificial Intelligence *(Introductory tier)*

**Facilitator:** Solomon Wilson MBCS | PhD Student, Computer Science | Deputy HOD Transport Planning & Operations | HOD, IT & Audit, SLPTA

**Mode:** Google Colab (zero-install)

**Mental model layer:** L11 — From Notebook to App

**Running scenario:** Route **R12** (Wilberforce → CBD) — operator OP-104, 25-minute delay
**Module:** 3 · **Week:** 4 · **Tier:** Intro

**New concept:** A notebook becomes an app when you add a UI around the model call

**Deliverable wired in:** None

## Learning objectives
By the end of today you will be able to:
- Explain what turns a notebook into an **application** (a UI around a model).
- Build a simple web interface with **Gradio** in a few lines.
- Wrap your Day 10 complaint classifier so a non-technical dispatcher can use it.

## Why this matters for SLPTA

A model that only runs in a notebook helps no one at the dispatch desk. **Gradio** wraps our complaint classifier in a web page: a dispatcher types a complaint, sees the category, and never touches code. This is the first time our work becomes a *tool* — TIA Lite.

## Environment setup

In [ ]:
# Gradio builds the web interface; google-genai powers the classifier.
!pip install -q gradio google-genai
print("Environment ready.")

Environment ready.


In [ ]:
# --- Standard SLPTA bootstrap (identical in every notebook) ----------------
import sys
from pathlib import Path
for candidate in [Path.cwd(), *Path.cwd().parents,
                  Path("/content/IPAM_USL_Intro_AI_5Week")]:
    if (candidate / "shared" / "slpta_bootstrap.py").exists():
        sys.path.insert(0, str(candidate / "shared"))
        break

from slpta_bootstrap import (MODEL, ensure_course_data, get_client,
                             load_route12_context, load_route_logs,
                             load_complaints, load_routes, load_operators)

ensure_course_data()
print("Model configured:", MODEL)
print(load_route12_context())

Model configured: gemini-2.0-flash
Route R12 (Wilberforce → CBD). The 07:45 service, operated by OP-104 on vehicle SLPTA-1142, departed 25 minutes late. Recorded cause: Heavy traffic on Wilkinson Road. About 40 passengers were affected and the dispatch desk received multiple complaints. (Synthetic SLPTA scenario — no real data.)


## API key reminder
This notebook calls Gemini, so you need your key. In Colab: click the **key icon** (Secrets) in the left sidebar, add a secret named **`GEMINI_API_KEY`**, paste your key, and toggle notebook access on.

<!-- cell-diagram:c07 -->
<p align="center"></p>

### Check your understanding (before running)
This cell connects to Gemini and defines a reusable classify helper.

**Predict:** Will the helper return one short category string for a routine R12 delay complaint?

In [ ]:
# Connect to Gemini and define a tiny helper we reuse all session.
client = get_client()

def ask(prompt):
    """Send a prompt to Gemini and return the text reply."""
    return client.models.generate_content(model=MODEL, contents=prompt).text

print("Connected to", MODEL)

Connected to gemini-2.0-flash


## Concept — first principles

An **application** = a model (or pipeline) + a **user interface** + input/output handling. The pattern is always:

> user input → our function → model → result shown back

**Gradio** gives us that UI almost for free: we write one Python function, hand it to `gr.Interface`, and call `.launch()`. Gradio renders the page inside Colab.

*Jargon, defined once:* a **pipeline** is the fixed sequence of steps from raw input to final output.

<p align="center"></p>

<!-- cell-diagram:c11 -->
<p align="center"></p>

### Check your understanding (before running)
The `classify_complaint` function will call Gemini with a fixed prompt. Before you run:
- What **category** do you expect for a complaint about a speeding R12 driver?
- What happens if the complaint text is empty?

*Write your predictions, then run.*

In [ ]:
def classify_complaint(text):
    """
    Classifies a passenger complaint into a specific category using the Gemini model.

    Args:
        text (str): The raw text of the passenger complaint.

    Returns:
        str: A single-word category or an error message if the API is unavailable.
    """
    if not text.strip():
        return "Please type a complaint."

    prompt = (
        "Classify this SLPTA passenger complaint into ONE of: Delay, Overcharging, "
        "Safety, Cleanliness, Staff Conduct, Lost Item, Other. Reply with only the "
        "category.\n\nComplaint: " + text
    )

    try:
        # Execution: Call the 'ask' helper function
        result = ask(prompt)
        return result.strip()
    except Exception as e:
        # Handle rate limits (Quota exceeded) or connection issues
        if "429" in str(e):
            return "Error: API Quota exceeded. Please wait 60 seconds and try again."
        return f"Error: {str(e)}"

# Quick test: Validate the classifier with a sample safety-related complaint
test_complaint = "Driver on R12 was speeding and overtaking dangerously"
print(f"Test Input: {test_complaint}")
print(f"Predicted Category: {classify_complaint(test_complaint)}")

Test Input: Driver on R12 was speeding and overtaking dangerously
Predicted Category: Error: API Quota exceeded. Please wait 60 seconds and try again.


<!-- cell-diagram:c13 -->
<p align="center"></p>

### Check your understanding (before running)
**Predict:** once we wrap the function in Gradio, what will a dispatcher need to know about Python to use it?

In [ ]:
import gradio as gr

# --- Gradio UI Configuration ---
# The gr.Interface class maps a Python function to a web UI.
demo = gr.Interface(
    # 'fn' connects the UI to our backend logic function defined above
    fn=classify_complaint,

    # 'inputs' defines the UI component where the user enters data
    inputs=gr.Textbox(label="Passenger complaint", lines=2, placeholder="Describe the issue here..."),

    # 'outputs' defines where the function's return value is displayed
    outputs=gr.Textbox(label="Predicted category"),

    # Visual branding and instructions for the end-user (Dispatchers)
    title="SLPTA Complaint Router (TIA Lite)",
    description="Type a passenger complaint; the assistant suggests a category for routing.",

    # 'examples' provides clickable presets to help users test the tool quickly
    examples=[
        ["Bus on R12 was 30 minutes late with no announcement"],
        ["Conductor charged me above the posted fare"],
        ["The R3 vehicle had a broken door and bald tyres"],
    ],
)

# --- App Launch ---
# .launch() starts the local web server.
# Note: share=True would generate a public URL for use outside this notebook.
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2431906b80e7f489a0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### Exercise — make it yours (change one thing)

Change the `title` below to your own (e.g. *"Dispatch Desk Assistant"*) and re-run to relaunch the app.

<!-- cell-diagram:c16 -->
<p align="center"></p>

### Check your understanding (before running)
You are about to change the `app_title` string and relaunch the Gradio app.
- Will the old app URL still work after you relaunch?
- What is the *only* thing a dispatcher needs to know to use this tool?

*Write your answers, then make the change and run.*

In [ ]:
# --- Exercise: Customizing the Application ---

# TODO: change the app title, then re-run to relaunch.
# This variable sets the main heading shown at the top of your web app.
app_title = "SLPTA Complaint Router (TIA Lite)"   # <-- change me

# Define the interface using our classification function
# fn: the logic, inputs: the textbox for text, outputs: the display for the category
gr.Interface(
    fn=classify_complaint,
    inputs=gr.Textbox(label="Passenger complaint", lines=2),
    outputs=gr.Textbox(label="Predicted category"),
    title=app_title,
).launch() # launch() starts the web server and provides the interface link

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f76c75fea30b41131f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Check your understanding
1. In your own words, what is the difference between a notebook and an application?
2. What does the `fn` argument to `gr.Interface` do?
3. Why does wrapping the model in a UI matter for SLPTA dispatch staff specifically?

### Your answers
*Double-click to edit this cell and type your answers here.*

1.
2.
3.

## If you remember one thing today…

> **An application is a model plus a door users can walk through — Gradio builds that door in a few lines.**

## Submission checklist
- [ ] Run every code cell successfully (top to bottom)
- [ ] Complete the exercise (fill every blank / make the requested change)
- [ ] Answer the **Check your understanding** questions in the markdown cell provided
- [ ] Save a clean copy of the notebook (*File → Save a copy in Drive*)